In [1]:
import pandas as pd

file_path = "../data/processed/dataset_final_qwen_filled.csv" 
df = pd.read_csv(file_path, dtype={'article_id': str})

df.info()

missing_data = df.isnull().sum()
print(missing_data[missing_data > 0])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70197 entries, 0 to 70196
Data columns (total 29 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   article_id                    70197 non-null  object
 1   product_code                  70197 non-null  int64 
 2   prod_name                     70197 non-null  object
 3   product_type_no               70197 non-null  int64 
 4   product_type_name             70197 non-null  object
 5   product_group_name            70197 non-null  object
 6   graphical_appearance_no       70197 non-null  int64 
 7   graphical_appearance_name     70197 non-null  object
 8   colour_group_code             70197 non-null  int64 
 9   colour_group_name             70197 non-null  object
 10  perceived_colour_value_id     70197 non-null  int64 
 11  perceived_colour_value_name   70197 non-null  object
 12  perceived_colour_master_id    70197 non-null  int64 
 13  perceived_colour

-> không có missing data, chỉ có 29 sản phẩm bị thiếu description -> gần như là 0%

In [2]:
# Lấy toàn bộ các cột có kiểu dữ liệu là chuỗi/text (object)
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Loại bỏ ID và các cột chứa đoạn văn dài không dùng để thống kê cardinality
exclude_cols = ['article_id', 'prod_name', 'detail_desc', 'refined_description']
target_cols = [col for col in categorical_cols if col not in exclude_cols]

print(f"Tổng số cột phân loại sẽ EDA: {len(target_cols)}")

for col in target_cols:
    unique_count = df[col].nunique()
    top_values = df[col].value_counts().head(5).to_dict()
    print(f"--- {col} ---")
    print(f"Unique: {unique_count}")
    print(f"Top 5: {top_values}\n")

Tổng số cột phân loại sẽ EDA: 15
--- product_type_name ---
Unique: 113
Top 5: {'Dress': 7815, 'Trousers': 6967, 'Sweater': 6408, 'T-shirt': 4441, 'Blouse': 3518}

--- product_group_name ---
Unique: 15
Top 5: {'Garment Upper body': 29397, 'Garment Lower body': 12383, 'Garment Full body': 8424, 'Accessories': 8096, 'Underwear': 4575}

--- graphical_appearance_name ---
Unique: 30
Top 5: {'Solid': 37862, 'All over pattern': 9479, 'Melange': 4524, 'Denim': 3081, 'Stripe': 2861}

--- colour_group_name ---
Unique: 50
Top 5: {'Black': 19000, 'White': 6222, 'Dark Blue': 5855, 'Light Beige': 2531, 'Light Pink': 2458}

--- perceived_colour_value_name ---
Unique: 8
Top 5: {'Dark': 31445, 'Dusty Light': 12926, 'Light': 9432, 'Medium Dusty': 8632, 'Bright': 4306}

--- perceived_colour_master_name ---
Unique: 20
Top 5: {'Black': 18955, 'Blue': 9520, 'White': 8203, 'Grey': 4815, 'Pink': 4616}

--- department_name ---
Unique: 167
Top 5: {'Jersey': 4587, 'Knitwear': 3496, 'Trouser': 2642, 'Blouse': 2354

Ta sẽ chia data ra:

**Nhóm core semantic**: ghép vào text embedding
- product_type_name (113 unique) kiểu như dress, trouser,...
- colour_group_name (50 unique): Rất chi tiết (Dark Blue, Light Pink). SigLIP rất nhạy cảm với màu sắc. Giữ cột này, bỏ qua perceived_colour_master_name (vì quá chung chung) và perceived_colour_value_name để tránh câu văn lủng củng ("Dark Dark Blue"). Phần value name và master name sẽ được đưa vào payload để bổ sung thông tin nếu cần.
- index_name (6 unique): Chứa giới tính/đối tượng (Ladieswear, Menswear). SigLIP cần context này để phân biệt áo nam/nữ.

**Nhóm conditional semantic**: chỉ đưa vào khi có giá trị đặc thù
- graphical_appearance_name: Có tới 37,862 sản phẩm là "Solid" (>50%). Nếu nhét chữ "Solid" vào mọi vector, nó sẽ trở thành Stop-word làm loãng vector. Rule: Chỉ đưa vào câu nếu giá trị KHÁC "Solid" (vd: Stripe, Melange, Denim).
- fit (301 unique) & occasion (520 unique) & seasonality (96 unique): Dữ liệu rác (Unspecified/Other) chiếm từ 18k - 23k dòng. Chữ "All-Season" cũng chiếm 33k dòng. Rule: Bỏ qua toàn bộ các chữ "Unspecified", "Other", "All-Season", "Regular/Straight". Chỉ giữ lại các key có tính phân loại cao (Oversized, Fitted, Winter, Party...).

**Nhóm redundant**: chỉ đưa vào payload
- department_name, section_name, product_group_name, garment_group_name


In [7]:
for col in target_cols:
    unique_count = df[col].nunique()
    print(f"Cột: {col} | Tổng số unique: {unique_count}")
    print(f"{'-'*60}")
    
    val_counts = df[col].value_counts()
    
    if unique_count > 200:
        print(f"Top 200 unique")
        display_data = val_counts.head(200)
    else:
        print(f"Toàn bộ unique")
        display_data = val_counts
        
    # In ra dạng list để dễ quan sát bằng mắt
    for val, count in display_data.items():
        print(f" - {val}: {count}")

Cột: product_type_name | Tổng số unique: 113
------------------------------------------------------------
Toàn bộ unique
 - Dress: 7815
 - Trousers: 6967
 - Sweater: 6408
 - T-shirt: 4441
 - Blouse: 3518
 - Top: 3326
 - Shirt: 2579
 - Jacket: 2423
 - Vest top: 2352
 - Skirt: 2280
 - Shorts: 2277
 - Bra: 2199
 - Underwear bottom: 2049
 - Hoodie: 1338
 - Earring: 1149
 - Bag: 1137
 - Blazer: 1013
 - Swimwear bottom: 1001
 - Socks: 988
 - Cardigan: 895
 - Bikini top: 848
 - Leggings/Tights: 821
 - Other accessories: 787
 - Scarf: 735
 - Boots: 699
 - Sneakers: 644
 - Hat/beanie: 584
 - Hair/alice band: 568
 - Necklace: 530
 - Sunglasses: 481
 - Pyjama set: 440
 - Coat: 436
 - Jumpsuit/Playsuit: 429
 - Belt: 421
 - Sandals: 373
 - Swimsuit: 364
 - Polo shirt: 335
 - Cap/peaked: 252
 - Bodysuit: 247
 - Ring: 234
 - Other shoe: 231
 - Pyjama bottom: 209
 - Heeled sandals: 200
 - Hat/brim: 188
 - Pumps: 186
 - Underwear Tights: 176
 - Underwear body: 165
 - Bracelet: 163
 - Hair clip: 161
 - 

Phân tích:
- product_type_name: 113 unique -> nhưng có nhiều loại thuộc nhóm acessories như earring, belt, wallet, nipple cover,... -> từ khóa hơi ngách nhưng vẫn nên giữ lại vì giá trị cao, không cần llm relabel
- colour_group_name: 50 unique -> Có các màu trùng nghĩa như Light Blue và Blue, hoặc các màu nhiễu như Other Pink, Other Yellow, Unknown, Other. Giữ nguyên trong payload làm exact match, trong text template thì bỏ chữ other đi (kiểu như other red thì chỉ lấy red thôi). Bỏ qua bọn unknown hay other đứng một mình
- graphical_appearance_name: solid gần 38k, các giá trị khác như Check (1699), Stripe (2861), Lace (1327) rất tốt. Nhưng có những giá trị nhiễu hoặc sai chính tả/dư thừa như Other pattern, Mixed solid/pattern, Unknown -> Trong Text Template, chỉ nối thêm chuỗi khi giá trị KHÔNG nằm trong list ['Solid', 'Unknown', 'Other structure', 'Other pattern', 'Treatment']
- fit: Dữ liệu quá rác. Cùng là áo/quần rộng nhưng có tới hàng chục cách viết: Loose, Relaxed, Oversized, Loose/Fit, Loose/Relaxed, Loose/Baggy, Relaxed/Loose, v.v. Cùng là form ôm cũng bị phân mảnh: Skinny, Slim, Fitted, Tight/Fitted, Fitted/Slim -> llm relabeling
- occasion: Tương tự fit, có hàng trăm cách viết khác nhau cho cùng một dịp mặc (vd: Party, Evening, Night Out, Going out all đều là party). Dữ liệu cũng rất rác với nhiều giá trị Unspecified/Other -> llm relabeling
- seasonality: tương tự
